RNN - Erro dos pesos computados e usado somente durante a iteração

In [1]:
import numpy as np
from numpy import linalg as LA
import pandas as pd
import operator as op
import ipynbname
import math
import matplotlib.cm as cm
import optuna
from optuna.samplers import RandomSampler
from optuna.visualization import plot_parallel_coordinate
from optuna.visualization import plot_pareto_front
from optuna.importance import get_param_importances
from optuna.exceptions import TrialPruned
import matplotlib.pyplot as plt
from matplotlib.patches import Circle
import matplotlib as mpl
#from Testing.RTLO import *
from Functions.RLS import *
from Functions.Graphs import *
from sklearn.metrics import root_mean_squared_error as RMSE
from sklearn.metrics import mean_absolute_percentage_error as MAPE
FileName = ipynbname.name()

params = [14, 12, 2, 0.01, 1e-08, 0.0001, 2]
df = pd.read_csv(r'Dataset\Bearing1_1.csv')
sig = df['PC1'].values

def prepare_data(sig, n, m):
    X, Y = [], []
    # O shift (s) é calculado para alinhar o final de Y com a predição futura
    # Seguindo sua lógica: se n=4, m=3 -> Y começa no índice 2 (hi_3)
    s = n - m + 1 
    
    for i in range(len(sig) - n - 1):
        X.append(sig[i : i + n])
        Y.append(sig[i + s : i + s + m])
        
    return np.array(X), np.array(Y)

def PlotPredError(rtlo,w=9,h=3):
    s = len(rtlo.yWAPE)
    t = rtlo.t
    fig, axes = plt.subplots(nrows=1, ncols=4, figsize=(w, h))
    axes = axes.flatten()
    ax1,ax2,ax3,ax4 = axes[0], axes[1], axes[2], axes[3]

    ax1.plot(t, rtlo.yR, color='black',label='Y-Real', linestyle='-')
    ax1.plot(t, rtlo.yP, color='blue',label='Y-Pred', linestyle='-')
    ax1.plot(t, rtlo.yL, color='blue', linestyle='--')
    ax1.plot(t, rtlo.yU, color='blue', linestyle='--')
    
    ax1.set_title('Y - Real x Prediction')
    ax1.set_xlabel('X')
    ax1.set_ylabel('Y', color='black')
    ax1.legend()

    ax2.plot(t, rtlo.eR, color='black',label='e-Real', linestyle='-')
    ax2.plot(t, rtlo.eP, color='blue',label='e-Pred', linestyle='-')
    ax2.set_title('Error - Real x Prediction')
    ax2.set_xlabel('X')
    ax2.set_ylabel('Prediction Error', color='black') 
    ax2.legend()
    
    ax3.plot(t[-s:], rtlo.yWAPE, color='blue',label='WAPE', linestyle='-')
    ax3.set_title('Prediction WAPE')
    ax3.set_xlabel('X')
    ax3.set_ylabel('WAPE', color='black') 

    '''ax4.plot(t[-s:], rtlo.rWAPE, color='blue',label='WAPE', linestyle='-')
    ax4.set_title('RUL Prediction WAPE')
    ax4.set_xlabel('X')
    ax4.set_ylabel('WAPE', color='black') '''

    fig.tight_layout()  # otherwise the right y-label is slightly clipped
    plt.show()


def PlotPredErrorPLY(rtlo, w=1200, h=300):
    # s: tamanho do vetor WAPE (caso comece depois do início)
    s = len(rtlo.yWAPE)
    t = rtlo.t
    
    # Criando o layout de 1 linha e 4 colunas
    fig = make_subplots(
        rows=1, cols=5, 
        shared_xaxes=True,
        subplot_titles=('Y - Real x Prediction', 'RUL - Real x Pred', 'Error - Real x Pred', 'Prediction WAPE', 'RUL Prediction WAPE')
    )

    # --- Subplot 1: Y Real x Pred (com Intervalos) ---
    fig.add_trace(go.Scatter(x=t, y=rtlo.yR, name='Y-Real', line=dict(color='black')), row=1, col=1)
    fig.add_trace(go.Scatter(x=t, y=rtlo.yP, name='Y-Pred', line=dict(color='blue')), row=1, col=1)
    # Intervalos (Dashed)
    fig.add_trace(go.Scatter(x=t, y=rtlo.yL, name='y-Lower', line=dict(color='blue', dash='dash'), showlegend=False), row=1, col=1)
    fig.add_trace(go.Scatter(x=t, y=rtlo.yU, name='y-Upper', line=dict(color='blue', dash='dash'), showlegend=False), row=1, col=1)

    fig.add_trace(go.Scatter(x=t, y=rtlo.rulR, name='rul R', line=dict(color='black'), showlegend=False), row=1, col=2)
    fig.add_trace(go.Scatter(x=t, y=rtlo.rulP, name='rul P', line=dict(color='blue'), showlegend=False), row=1, col=2)
    fig.add_trace(go.Scatter(x=t, y=rtlo.rulL, name='rul L', line=dict(color='blue'), showlegend=False), row=1, col=2)
    fig.add_trace(go.Scatter(x=t, y=rtlo.rulU, name='rul U', line=dict(color='blue'), showlegend=False), row=1, col=2)

    # --- Subplot 2: Error Real x Prediction ---
    fig.add_trace(go.Scatter(x=t, y=rtlo.eR, name='e-Real', line=dict(color='black')), row=1, col=3)
    fig.add_trace(go.Scatter(x=t, y=rtlo.eP, name='e-Pred', line=dict(color='blue')), row=1, col=3)

    # --- Subplot 3: Prediction WAPE ---
    fig.add_trace(go.Scatter(x=t[-s:], y=rtlo.yWAPE, name='WAPE', line=dict(color='blue')), row=1, col=4)

    # --- Subplot 4: RUL Prediction WAPE (Habilitado se existir) ---
    if hasattr(rtlo, 'rWAPE'):
        fig.add_trace(go.Scatter(x=t[-len(rtlo.rWAPE):], y=rtlo.rWAPE, name='RUL-WAPE', line=dict(color='red')), row=1, col=5)

    # Atualizando Layout e Eixos
    fig.update_layout(
        width=w, height=h,
        title_text=f"RTLO Model Performance Analysis",
        template='plotly_white',
        showlegend=True,
        margin=dict(l=40, r=40, t=80, b=40)
    )

    # Labels dos eixos (opcional, já que os títulos ajudam)
    fig.update_xaxes(title_text="Time / Index")
    fig.update_yaxes(title_text="Amplitude", col=1)
    fig.update_yaxes(title_text="Error", col=2)
    fig.update_yaxes(title_text="WAPE (%)", col=3)

    fig.show()
    

In [53]:
def prepare_data(sig, n, m):
    X, Y = [], []
    s = n - m + 1 
    for i in range(len(sig)-n-m+1):
        X.append(sig[i : i + n])
        Y.append(sig[i + n : i +n+ m])
    return np.array(X), np.array(Y)

X,Y = prepare_data(sig,3,3)
print(sig[-5:])
print(X[-1])
print(Y[-1])

[0.22760718 0.21406503 0.19205265 0.1757025  0.165151  ]
[0.24015307 0.22760718 0.21406503]
[0.19205265 0.1757025  0.165151  ]


In [67]:
def TANH(x):
    return np.tanh(x)
    #return 1 / (1 + np.exp(-x))
    #return np.maximum(0, x)

def dTANH(x):
    #return 1/np.cosh(10*np.tanh(x/10))**2  # the tanh prevents oveflow
    return 1-(TANH(x)**2)
    #return TANH(x)*(1-TANH(x))
    #return np.where(x > 0, 1, 0)

def split_matrix(A):
    Ap = np.where(A > 0, A, 0)
    An = np.where(A < 0, A, 0)
    return An, Ap

def get_interval(A):
    v_max = np.max(A, axis=0)
    v_min = np.min(A, axis=0)
    
    return v_min, v_max

def XavierUniform(shape,sd):
    np.random.seed(sd)
    n_in, n_out = shape
    limit = np.sqrt(6 / (n_in + n_out))
    return np.random.uniform(-limit, limit, size=shape) 

class RTLO:
    def __init__(self, nI,nR,nO,ηS=[0.1,0.1,0.1], τ=10,lr=1e-5):
        np.random.seed(42)
        self.k = 1
        self.j = nI-1
        self.t = np.array([])
        self.ref = None

        self.nI = nI
        self.nR = nR
        self.nO = nO

        self.ηS = np.array(ηS)
        self.τ = τ
        self.ρ = 0.1

        self.xPi = np.zeros(nI)
        self.xUi = np.zeros(nI)
        self.xLi = np.zeros(nI)

        self.hP = np.zeros(nR)
        self.hU = np.zeros(nR)
        self.hL = np.zeros(nR)
        self.z = np.zeros(nR)

        self.htUF = np.zeros(nR)
        self.htUF = np.zeros(nR)
        self.uP = np.zeros(nR)
        self.decay = lr
        
        self.pS = np.zeros((self.nR, self.nR))
        self.qS = np.zeros((self.nR, self.nI))

        self.p = np.zeros((self.nR, self.nR, self.nR))
        self.q = np.zeros((self.nR, self.nR, self.nI))

        self.ΔOS = np.zeros((nO, nR))
        self.ΔRS = np.zeros((nR, nR))
        self.ΔIS = np.zeros((nR, nI))
        
        self.wI = XavierUniform([nR, nI],sd=42)
        self.wR = XavierUniform([nR, nR],sd=41)
        self.wO = XavierUniform([nO, nR],sd=40)
        self.BS = XavierUniform([nR, nO],sd=39)
        #self.BS = np.random.randn(nR, nO)/nO**0.5

        '''self.wI = 0.1*(np.random.randn(nR, nI)-1)
        self.wR = 1.5*(np.random.randn(nR, nR)/nR**0.5)
        self.wO = 0.1*(2*np.random.randn(nO, nR)-1)/nR**0.5
        self.BS = np.random.randn(nR, nO)/nO**0.5'''

        self.rls = RLS_LogarithmicRegressor(0.9,1e7)
        
        self.yP = np.array([])
        self.yR = np.array([])
        self.yL = np.array([])
        self.yU = np.array([])

        self.eS = np.zeros(nI)
        self.eP = np.array([])
        self.eR = np.array([])

        self.εY = 0
        self.εR = 0
        self.εE = 0
        self.ΣW = 0

        self.μrWAPE = 0
        self.MPsum = 0

        self.yWAPE = np.array([])
        self.rWAPE = np.array([])
        self.eWAPE = np.array([])

        self.rR = 1e-10
        self.rP = 1e-10
        self.rL = 1e-10
        self.rU = 1e-10
        self.rRsum = 0

        self.rulR = np.array([])
        self.rulP = np.array([])
        self.rulL = np.array([])
        self.rulU = np.array([])

    def PredSingle(self,x):

        u = np.dot(self.wR, self.hS) + np.dot(self.wI, x)
        h = self.hS + (-self.hS + TANH(u))/self.τ
        y = np.dot(self.wO, h)

        return y

    def fit(self,xP,yR,start=0,store=False,show=False):

        exp=2
        k = self.k
        W = self.j**exp
        τ = self.τ
        BS = self.BS

        nR,nI,nO = self.nR, self.nI, self.nO
        wR,wI,wO = self.wR,self.wI,self.wO
        hPi, uPi = self.hP, self.uP
        pS, qS = self.pS, self.qS
        p,q = self.p, self.q
        
        η1,η2,η3 = self.ηS
        xPi = self.xPi

        uS = wR@hPi + wI@xP
        hP = hPi + (-hPi + TANH(uS))/τ
        yP = wO@hP
        eS = yR-yP

        '''print("hPi:",hPi)
        #print("hP:",hP)

        print("uS",uS)
        print("wO:",wO)


        print('yR:',yR)
        '''
        if show: 
            print('yR:',yR)
            print('yP:',yP)
        pS = np.outer(dTANH(uS),hPi)/τ + (1-1/τ)*pS
        qS = np.outer(dTANH(uS),xPi)/τ + (1-1/τ)*qS


        '''p = np.tensordot((1-1/τ)*np.eye(nR)
                    + dTANH(uS)*wR/τ, p, axes=1)
        q = np.tensordot((1-1/τ)*np.eye(self.n_rec)
                    + dTANH(uS)*wR/τ, q, axes=1)
        
        for i in range(nR):
            p[i, i, :] += df(u[tt+1, jj])*hP/τ
            q[i, i, :] += df(u[tt+1, jj])*xP/τ'''

        δOS = η1*np.outer(eS,hP)
        δRS = η2*np.outer((BS@eS),np.ones(nR))*pS
        δIS = η3*np.outer(np.dot(BS, eS),np.ones(nI))*qS

        self.ΔOS = (self.ΔOS*(self.k-1) + δOS)/self.k
        self.ΔRS = (self.ΔRS*(self.k-1) + δRS)/self.k
        self.ΔIS = (self.ΔIS*(self.k-1) + δIS)/self.k

        self.wI = self.wI + δIS
        self.wR = self.wR + δRS
        self.wO = self.wO + δOS
        self.pS = pS
        self.qS = qS

        self.hP = hP
        self.xPi = xP

        self.UpdateRLS(yP,yR)

        ΣW = self.ΣW + W
        ΔY = np.abs((yR-yP)/yR)[-1]
        ΔR = np.abs((self.rR-self.rP)/(self.rR+1e-10))
        ΔE = np.abs((self.eR[-1]-self.eP[-1])/self.eR[-1])

        if k > start:
            self.εY = ((self.εY*self.ΣW) + (W*ΔY))/ΣW
            self.εR = ((self.εR*self.ΣW) + (W*ΔR))/ΣW
            self.εE = ((self.εE*self.ΣW) + (W*ΔE))/ΣW
            self.ΣW = ΣW

        if store:
            self.yR = np.append(self.yR,yR[-1])
            self.yP = np.append(self.yP,yP[-1])
            #self.yL = np.append(self.yL,yP[-1])
            #self.yU = np.append(self.yU,yP[-1])
            self.yWAPE = np.append(self.yWAPE,self.εY)
            self.rWAPE = np.append(self.rWAPE,self.εR)
            self.eWAPE = np.append(self.eWAPE,self.εE)

        self.k = self.k+1
        self.j = self.j+1
        self.t = np.append(self.t,k+nI)
        #self.ηS = self.ηS/(1 + self.decay*self.k)


    def PredRulIntr(self, x,lim=0.2,maxRul=110,store=False,show=False):
        xL,xP,xU =x.copy(), (x-(self.ρ*self.eS)).copy(),(x+(self.ρ*self.eS)).copy()        

        #print(xL[-5:])
        #print(xP[-5:])
        #print(xU[-5:])
        predict = True
        k=1
        PredRuls = [True for i in range(3)]
        PredVals, Ruls = [0 for i in range(3)], [0 for i in range(3)]
        #print(self.ht)
        wR,wI,wO = self.wR,self.wI,self.wO

        wRp, wRn = np.maximum(0, self.wR), np.abs(np.minimum(0, self.wR))
        wIp, wIn = np.maximum(0, self.wI), np.abs(np.minimum(0, self.wI))
        wOp, wOn = np.maximum(0, self.wO), np.abs(np.minimum(0, self.wO))
        #hU,hL = np.maximum(0, self.hS), np.minimum(0, self.hS)
        hL,hP,hU = [self.hP.copy() for i in range(3)]

        while predict:
            #print('hL:',hL[-5:])
            #print('hU:',hU[-5:])

            #print('xL:',xL[-3:],'xU:',xU[-3:])
            #print('xL:',xL[-3:],'xU:',xU[-3:])
            
            uP = wR@hP + wI@xP
            uL = (wRp @ hL - wRn @ hU) + (wIp @ xL - wIn @ xU)
            uU = (wRp @ hU - wRn @ hL) + (wIp @ xU - wIn @ xL)

            hP = hP*(1-1/self.τ) + TANH(uP)/self.τ
            hL = hL*(1-1/self.τ) + TANH(uL)/self.τ
            hU = hU*(1-1/self.τ) + TANH(uU)/self.τ
            
            yP = (wO@hP)[0]
            yL = (wOp @ hL - wOn @ hU)[0]
            yU = (wOp @ hU - wOn @ hL)[0]

        
            #print('hL:',hL[-5:])
            #print('hP:',hP[-5:])
            #print('hU:',hU[-5:])

            #print('xP:',xP[-3:],'yP:',yP[-4:])
            #print('xU:',xU[-3:],'yU:',yU)

            xP = np.delete(np.append(xP,yP),0)
            xL = np.delete(np.append(xL,yL),0)
            xU = np.delete(np.append(xU,yU),0)

            PredVals = [yL,yP,yU]

            if show:
                print(PredVals)
            

            if k == 1:
                self.yL = np.append(self.yL,yL)
                self.yU = np.append(self.yU,yU)
                #print(PredVals)
            k = k+1
            
            CheckPred,CheckLim=0,0

            for i in range(3):
                if PredRuls[i]: 
                    Ruls[i] = Ruls[i]+1
                    if Ruls[i] >= maxRul:
                        CheckLim = CheckLim + 1
                        Ruls[i] = maxRul
                        PredRuls[i] = False
                if PredVals[i] < lim: PredRuls[i] = False
                if not PredRuls[i]: CheckPred = CheckPred + 1
            if CheckPred == 3:break
            if CheckLim == 3:break
        
        self.rR=self.ref-self.k
        self.rL,self.rP,self.rU = Ruls

        if store:
            self.rulR = np.append(self.rulR,self.rR)
            self.rulL = np.append(self.rulL,self.rL)
            self.rulP = np.append(self.rulP,self.rP)
            self.rulU = np.append(self.rulU,self.rU)
    
    def PredRul(self, x,lim=0.2,store=False):
        xP = x.copy()
        rulP=0
        predict = True
        wR,wI,wO = self.wR,self.wI,self.wO

        hS = self.hP
        while predict:
            u = (wR @ hS) + (wI @ xP)
            hS = hS + (1/self.τ) * (-hS + TANH(u))
            yP = (wO @ hS)[-1]
            #print(yP)
            xP = np.delete(np.append(xP,yP),0)

            if predict:
                rulP = rulP+1

            if yP < lim:
                predict = False

            if rulP >= 110:
                rulP = 0
                break

        self.rR=self.ref-self.k
        self.rP = rulP

        if store:
            self.rulR = np.append(self.rulR,self.rR)
            self.rulP = np.append(self.rulP,self.rP)
    
    def UpdateRLS(self,yP,yR):
        eP = np.abs(self.rls.predict(np.abs(yP[-1])))
        eR = np.abs(yP-yR)[-1]
        self.rls.update(np.abs(yP[-1]), eR)
        self.eR = np.append(self.eR,eR)
        self.eP = np.append(self.eP,eP)
        self.eS = np.append(self.eS,eP)
        self.eS = np.delete(self.eS,0)

    


In [77]:
rates = [1/(10**i) for i in range(2,8)][::-1]

def objective(trial):

    nI = trial.suggest_int('nI', 2, 30) 
    nR = trial.suggest_int('nR', 1, 30) 
    nO = trial.suggest_int('nO', 1, 30) 
    N1 = trial.suggest_categorical('N1', rates) 
    N2 = trial.suggest_categorical('N2', rates) 
    N3 = trial.suggest_categorical('N3', rates) 
    τ = trial.suggest_int('τ', 1, 30)
    ηS = [N1,N2,N3]
        
    X,Y = prepare_data(sig,n=nI,m=nO)

    rtlo = RTLO(nI,nR,nO,ηS,τ)
    rtlo.ref = len(sig)-nI

    for i,_ in enumerate(X):
        #rtlo.PredRul(x=X[i],store=True)
        rtlo.fit(X[i],Y[i],start=0,store=False)

        '''if i == 50:
            if np.mean(rtlo.rulP)<np.mean(rtlo.rulR)*0.3:
                raise TrialPruned()
            
        if i == 85:
            if np.mean(rtlo.rulP[50:])>np.mean(rtlo.rulR[50:])*1.5:
                raise TrialPruned()'''
            
            #if np.mean(rtlo.rulP)> np.mean(rtlo.rulR)*1.15:
            #        raise TrialPruned()

    return rtlo.εY


#sampler = optuna.samplers.TPESampler(multivariate=True,group=True,n_startup_trials=2000)
sampler=RandomSampler()
study = optuna.create_study(
    direction="minimize",
    sampler=sampler,
    #pruner=pruner
    #storage="sqlite:///" + f'Optuna/{FileName}_Prdct.db', study_name=f'P{4}',
    load_if_exists=True)
study.optimize(objective, n_trials=5000)
best_params = study.best_params
params = list(best_params.values())
print('Erro:', study.best_value, 'parameters: ', params)

[I 2026-04-14 16:38:06,541] A new study created in memory with name: no-name-b6c7dabb-c502-45bd-ad4b-77c5b66cd762
[I 2026-04-14 16:38:06,559] Trial 0 finished with value: 1.6411701702856805 and parameters: {'nI': 10, 'nR': 26, 'nO': 3, 'N1': 1e-05, 'N2': 1e-05, 'N3': 0.001, 'τ': 9}. Best is trial 0 with value: 1.6411701702856805.
[I 2026-04-14 16:38:06,567] Trial 1 finished with value: 1.9308334392355315 and parameters: {'nI': 8, 'nR': 26, 'nO': 28, 'N1': 1e-06, 'N2': 0.0001, 'N3': 0.0001, 'τ': 1}. Best is trial 0 with value: 1.6411701702856805.
[I 2026-04-14 16:38:06,575] Trial 2 finished with value: 2.821875739407955 and parameters: {'nI': 2, 'nR': 15, 'nO': 6, 'N1': 1e-05, 'N2': 0.01, 'N3': 0.001, 'τ': 1}. Best is trial 0 with value: 1.6411701702856805.
[I 2026-04-14 16:38:06,588] Trial 3 finished with value: 0.4508876160128201 and parameters: {'nI': 2, 'nR': 6, 'nO': 4, 'N1': 1e-06, 'N2': 1e-05, 'N3': 1e-05, 'τ': 1}. Best is trial 3 with value: 0.4508876160128201.
[I 2026-04-14 16:

KeyboardInterrupt: 

Erro: 0.019092593340711093 parameters:  [4, 20, 1, 0.01, 0.01, 1e-05, 2]\



In [70]:
nI,nR,nO,N1,N2,N3,τ= params
ηS = [N1,N2,N3]
X,Y = prepare_data(sig,n=nI,m=nO)
rnn = RTLO(nI,nR,nO,ηS,τ)
rnn.ref = len(sig)-nI
for i in range(len(X[:])):
    #rnn.PredRul(x=X[i],store=True)
    rnn.PredRulIntr(x=X[i],store=True)
    rnn.fit(X[i],Y[i],start=0,store=True)
#print(.wR[0])


print(rnn.εY)  
print(rnn.rWAPE[-1])      
PlotPredErrorPLY(rnn,w=1500)


0.019092593340711093
244891746.8698239


In [157]:
i=75
r_m = np.mean(rnn.rulR[:i])
r_mL = np.mean(rnn.rulR[:i])*0.3
r_mU = np.mean(rnn.rulR[:i])*1.15
p_m = (np.mean(rnn.rulP[:i]))

print('lower:',r_mL,'mid:',r_m,'upper:',r_mU)
print('pred:',p_m)

i=50
f=90
r_m = np.mean(rnn.rulR[i:f])
r_mL = np.mean(rnn.rulR[i:f])*0.3
r_mU = np.mean(rnn.rulR[i:f])*1.5
p_m = (np.mean(rnn.rulP[i:f]))

print('lower:',r_mL,'mid:',r_m,'upper:',r_mU)
print('pred:',p_m)

lower: 21.3 mid: 71.0 upper: 81.64999999999999
pred: 71.64
lower: 11.549999999999999 mid: 38.5 upper: 57.75
pred: 48.95


In [379]:
#params =  [14, 8, 14, 0.001, 0.01, 1e-06, 1e-07, 29]
nI,nR,nO,N1,N2,N3,τ= params
ηS = [N1,N2,N3]
X,Y = prepare_data(sig,n=nI,m=nO)
rnn = RTLO(nI,nR,nO,ηS,τ)
rnn.ref = len(sig)-nI

i=0

In [382]:
#rnn.PredRul(x=X[i],store=True)
rnn.PredRulIntr(x=X[i],store=True,show=True)

rnn.fit(X[i],Y[i],start=0,store=True,show=False)
i=i+1

[0.23516146606492674, 0.23583983655887092, 0.23801385133223785]
[0.29148142555090145, 0.29465898961424963, 0.297994825260674]
[0.34823702429932174, 0.35112360147116406, 0.3580020704303819]
[0.3808669407322221, 0.3870751298028704, 0.3952649270293022]
[0.4194512745764764, 0.42935474190030704, 0.44017063281057683]
[0.4570607267400093, 0.46902121591166723, 0.4860884481382543]
[0.46830220563487673, 0.4847069716333589, 0.5085406857587531]
[0.4655357315968584, 0.48982713057177074, 0.5214997362421201]
[0.4579711903221976, 0.49597520082401964, 0.5365620820365714]
[0.46797718674255406, 0.5235756221462885, 0.5748760311466832]
[0.5009179552717841, 0.5742859297321989, 0.6411674231440986]
[0.5269896306495099, 0.6223691608017013, 0.7070056270920381]
[0.549476612935753, 0.6692935838940388, 0.7794980444849124]
[0.5242549200356795, 0.6855974906772175, 0.8336028302288878]
[0.47108050850537786, 0.6864773397322742, 0.8785115178410589]
[0.3851844613439874, 0.6772654938912609, 0.927396363507172]
[0.283185809